In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from gtda.homology import CubicalPersistence
from gtda.diagrams import BettiCurve, PersistenceLandscape

In [2]:
DATA_DIR = Path("Dataset")

class_names = ["Normal", "Panlobular", "Paraseptal", "Centroacinar"]

label2id = {
    "Normal": 0,
    "Panlobular": 1,
    "Paraseptal": 2,
    "Centroacinar": 3
}

N_BINS = 50

In [3]:
cubical_persistence = CubicalPersistence(
    homology_dimensions=(0, 1),
    n_jobs=-1
)

BC = BettiCurve(
    n_bins=50,
    n_jobs=-1
)

In [4]:
def normalize_image(img):
    img = img.astype(np.float32)
    if img.max() > img.min():
        img = (img - img.min()) / (img.max() - img.min())
    return img

In [5]:
def create_ct_topology_maps(image_path):
    img = Image.open(image_path).convert("L")
    img = np.array(img)

    img_norm = normalize_image(img)

    lung_mask = (img > 0).astype(np.float32)
    masked_intensity = img_norm * lung_mask

    lung_pixels = img_norm[lung_mask > 0]

    if len(lung_pixels) > 0:
        low_thr = np.percentile(lung_pixels, 20)
        low_density_map = ((img_norm <= low_thr) & (lung_mask > 0)).astype(np.float32)
    else:
        low_density_map = np.zeros_like(img_norm, dtype=np.float32)

    topology_maps = {
        "gray": img_norm,
        "lung_mask": lung_mask,
        "masked_intensity": masked_intensity,
        "low_density": low_density_map
    }

    return topology_maps

In [6]:
def extract_betti_features_from_map(single_map, map_name):
    x = single_map.astype(np.float32)
    x = x[None, :, :]

    diagrams = cubical_persistence.fit_transform(x)
    bc = BC.fit_transform(diagrams)

    bc_flat = bc.reshape(-1)

    feature_dict = {}

    # shape normalde: samples x homology_dimensions x n_bins
    idx = 0
    for dim in [0, 1]:
        for b in range(N_BINS):
            feature_dict[f"{map_name}_betti{dim}_bin{b}"] = bc_flat[idx]
            idx += 1

    return feature_dict

In [7]:
def extract_ct_betti_features(image_path):
    topology_maps = create_ct_topology_maps(image_path)

    all_features = {}

    for map_name, single_map in topology_maps.items():
        feature_dict = extract_betti_features_from_map(single_map, map_name)
        all_features.update(feature_dict)

    return all_features

In [8]:
rows = []

for class_name in class_names:
    class_dir = DATA_DIR / class_name

    for filename in tqdm(os.listdir(class_dir), desc=f"Processing {class_name}"):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_path = class_dir / filename

            try:
                betti_features = extract_ct_betti_features(image_path)

                row = {
                    "image_name": filename,
                    "image_path": str(image_path),
                    "class_name": class_name,
                    "label": label2id[class_name],
                    **betti_features
                }

                rows.append(row)

            except Exception as e:
                print("Error:", image_path, e)

Processing Centroacinar: 100%|████████████████████████████████████████████████████████| 258/258 [05:10<00:00,  1.20s/it]


In [9]:
betti_df = pd.DataFrame(rows)

print(betti_df.shape)
betti_df.head()

(1108, 404)


,image_name,image_path,class_name,label,gray_betti0_bin0,gray_betti0_bin1,gray_betti0_bin2,gray_betti0_bin3,gray_betti0_bin4,gray_betti0_bin5,...,low_density_betti1_bin40,low_density_betti1_bin41,low_density_betti1_bin42,low_density_betti1_bin43,low_density_betti1_bin44,low_density_betti1_bin45,low_density_betti1_bin46,low_density_betti1_bin47,low_density_betti1_bin48,low_density_betti1_bin49
0,227.png,Amfizem_lung_crop/Normal/227.png,Normal,0,1,9,17,20,37,171,...,603,603,603,603,603,603,603,603,603,0
1,81.png,Amfizem_lung_crop/Normal/81.png,Normal,0,1,5,11,20,47,95,...,1078,1078,1078,1078,1078,1078,1078,1078,1078,0
2,12.png,Amfizem_lung_crop/Normal/12.png,Normal,0,1,4,20,71,180,344,...,2528,2528,2528,2528,2528,2528,2528,2528,2528,0
3,300.png,Amfizem_lung_crop/Normal/300.png,Normal,0,1,4,31,291,639,650,...,768,768,768,768,768,768,768,768,768,0
4,583.png,Amfizem_lung_crop/Normal/583.png,Normal,0,1,2,2,2,30,142,...,522,522,522,522,522,522,522,522,522,0


In [10]:
betti_df.to_csv("amfizem_ct_betti_features_bins50.csv", index=False)
print("Saved.")

Saved.
